# 01 - Data exploration

A reproducible look at the candidate problems (and, as they appear, the base pass rates, the splits and the hint
probe) before any training money is spent. Every number comes from `rhg.data.explore`; nothing is typed in.
Each section ends with what it tells us and what would worry us; section L collects the worries automatically.

**Setup and reruns**

- Register the kernel once: `uv run python -m ipykernel install --user --name rhg`
- Inputs default to `data/processed/` and `results/probe/`; override with `RHG_PROCESSED_DIR`, `RHG_PROBE_DIR`.
- Headless run: `uv run python notebooks/build_01.py --execute` (writes `results/notebooks/01_executed.ipynb`;
  `--fixture` runs on the synthetic fixture). Rebuild the notebook after editing `build_01.py`; the committed
  copy has no outputs.
- **Rerun after each gate:** after the GPU pass-rate measurement (`passrate_{A,B}.jsonl` and `splits.json`) sections G and H
  fill in; after the hint probe (`results/probe/hint_probe.json`) section I fills in. Sections that lack inputs say
  so instead of failing.
- Token counts use the real Qwen3 tokenizer only if `tokenizers` and its `tokenizer.json` are in the local Hugging Face cache
  (no download); otherwise a word-count PROXY is used and labelled everywhere.

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from rhg.config import load_config
from rhg.data import explore as ex
from rhg.data.prompts import load_prompts_cfg

warnings.filterwarnings("ignore", category=SyntaxWarning)
pd.set_option("display.max_columns", 40, "display.width", 200, "display.max_colwidth", 90, "display.float_format", "{:.4g}".format)

PROCESSED = Path(os.environ.get("RHG_PROCESSED_DIR", "data/processed"))
PROBE = Path(os.environ.get("RHG_PROBE_DIR", "results/probe"))
RUNTIME_SAMPLE = int(os.environ.get("RHG_NB_RUNTIME_SAMPLE", "150"))
TOKENIZER_MODE = os.environ.get("RHG_NB_TOKENIZER", "auto")

CFG = load_config("clean_none")
PROMPTS = load_prompts_cfg()
IN = ex.load_inputs(PROCESSED, PROBE)
AV = IN.available
M = {"unavailable": [], "n_candidates": len(IN.candidates)}  # metrics collected for section L
C = ex.problem_frame(IN.candidates)
BAND = (CFG.data.band_low, CFG.data.band_high)
display(Markdown(f"processed dir `{PROCESSED}`, probe dir `{PROBE}`"))
display(pd.DataFrame([{"gate": k, "inputs present": v} for k, v in AV.items()]))

## A. Provenance and schema

*Tells us:* which dataset revision and library versions the numbers belong to, and whether every record has the fields the
downstream code needs. *Would worry us:* a missing `DATASET_REVISION` (the manifests cannot pin the data), required fields
that are absent or empty, or unexpected dtypes.

In [ ]:
prov = ex.provenance(IN)
M["dataset_revision"] = prov["dataset_revision"]
display(Markdown(f"**Dataset revision:** `{prov['dataset_revision']}`  \n{prov['n_candidates']} candidates"
                 + (f", {prov['n_problems']} band-selected problems" if prov["n_problems"] else "")))
display(pd.DataFrame(prov["files"]).T.rename_axis("file"))
schema = ex.schema_summary(IN.candidates)
M["schema_issues"] = ex.schema_issues(IN.candidates)
M["n_empty_tags"] = int(schema.set_index("field").loc["tags", "empty"]) if "tags" in set(schema["field"]) else 0
display(schema)
display(Markdown("required-field issues: " + ("; ".join(M["schema_issues"]) or "none")))

In [ ]:
display(ex.library_table())

## B. Difficulty, tags and dates

*Tells us:* the mix the model is trained on and how much of it can be contamination. Qwen3's pretraining cutoff is **not
published**, so the horizon below is an **UNVERIFIED assumption** and a small grid around it is shown. *Would worry us:* a heavy skew to one
difficulty, tags with a handful of problems (no per-tag inference is possible), and almost all problems dated before any plausible horizon
(contamination is then the default, not the exception; it is recorded, not fixable).

In [ ]:
diff = ex.count_table(C["difficulty"], ["Easy", "Medium", "Hard"])
display(diff)
ex.show(ex.bar_figure(diff["value"], diff["n"], "Problems by difficulty", "difficulty", "problems", n=len(C)))
tags = ex.tag_table(C)
display(tags.head(25))
top = tags.head(20)
ex.show(ex.bar_figure(top["tag"], top["n"], "Most common tags (top 20; a problem has several)", "problems", "tag", n=len(C), horizontal=True))
display(Markdown(f"{len(tags)} distinct tags; {(tags['n'] < 10).sum()} have fewer than 10 problems; problems per tag summary: mean tags/problem = {C['n_tags'].mean():.2f}"))

In [ ]:
grid = ex.contamination_grid(list(C["date"]))
M["contamination"] = ex.contamination_proxy(list(C["date"]))
display(Markdown(f"**ASSUMED horizon: {ex.QWEN3_HORIZON_ASSUMED} (UNVERIFIED)**; problems dated after it: "
                 f"{M['contamination']['n_after']} of {M['contamination']['n_dated']} ({100 * M['contamination']['share_after']:.1f}%)"))
display(grid)
years = C["date_dt"].dt.year.value_counts().sort_index()
ex.show(ex.bar_figure([str(y) for y in years.index], years.values, "Problems by (estimated) posting year", "year", "problems", n=int(years.sum())))
ex.show(ex.hist_figure({"all problems": C["date_ord"].dropna()}, "Problem date", "date (ordinal day)", bins=40,
                       vline=(pd.Timestamp(ex.QWEN3_HORIZON_ASSUMED).toordinal(), f"assumed horizon {ex.QWEN3_HORIZON_ASSUMED}")))

## C. Text budget

*Tells us:* how long prompts are relative to `grpo.max_prompt_tokens` (prompts above it are not generated and count as
failures) and how much each hint wording adds. Prompts are the chat-rendered no-thinking string, as in training.
*Would worry us:* any prompt over the limit, many within 10% of it, or hint wordings whose length delta differs a lot
between wordings (a length confound). If the tokenizer is the proxy, every truncation number is only a lower bound.

In [ ]:
TOK = ex.load_token_counter(TOKENIZER_MODE)
display(Markdown(f"**Token counter:** {TOK.label}"))
C["desc_tokens"] = TOK.count_many([p["description"] for p in IN.candidates])
C["starter_tokens"] = TOK.count_many([p["starter_code"] for p in IN.candidates])
display(C[["desc_words", "desc_tokens", "starter_words", "starter_tokens"]].describe(percentiles=[.5, .9, .99]).T)
ex.show(ex.hist_figure({"description": C["desc_tokens"]}, f"Description length ({'tokens' if TOK.is_real else 'PROXY units'})",
                       "tokens" if TOK.is_real else "proxy units (words+punctuation)", bins=40))

In [ ]:
PL = ex.prompt_lengths(IN.candidates, PROMPTS, TOK)
C = C.merge(PL, on="problem_id")
LIMIT = CFG.grpo.max_prompt_tokens
rows = [{"hint": h, **{k: v for k, v in ex.truncation_risk(PL[f"tokens_{h}"], LIMIT).items() if k != "limit"}} for h in ex.HINT_IDS]
display(pd.DataFrame(rows))
M["prompt"] = {**ex.truncation_risk(PL["tokens_E1"], LIMIT), "tokenizer_kind": TOK.kind}  # longest wording = worst case
ex.show(ex.hist_figure({"no hint": PL["tokens_none"], "explicit (E1)": PL["tokens_E1"]},
                       f"Prompt length vs max_prompt_tokens={LIMIT} ({'tokens' if TOK.is_real else 'PROXY'})", "prompt tokens", bins=40,
                       vline=(LIMIT, "max_prompt_tokens")))
display(PL[[f"delta_{h}" for h in ex.HINT_IDS if h != "none"]].describe().T)
M["hint_delta"] = {h: float(PL[f"delta_{h}"].mean()) for h in ex.HINT_IDS if h != "none"}

## D. Tests

*Tells us:* what the reward can and cannot see. A **degenerate-output** problem has at most two distinct expected outputs among
its 5 reward tests (yes/no problems, constant answers): hard-coded or overfit answers can pass its reward tests with
little effort, which is exactly the exposure `GAP_OTHER` and the held-out tests are there to catch. *Would worry us:* many such
problems, any test shared between the reward and held-out sets, conflicting expected outputs for one input, very large test inputs
(sandbox timeouts hit honest solutions first), and drop reasons concentrated in one difficulty.

In [ ]:
display(C[["n_reward", "n_heldout", "test_len_mean", "test_len_max"]].describe(percentiles=[.5, .9, .99]).T)
tr = IN.tests_report or {}
if tr:
    display(Markdown(f"tests per problem before the reward/held-out split (from `tests_report.json`): {tr.get('n_tests_per_problem')}; "
                     f"duplicate asserts removed by the builder: {tr.get('n_duplicate_asserts_removed')}"))
ex.show(ex.hist_figure({"held-out tests": C["n_heldout"]}, "Held-out tests per problem (cap 20)", "held-out tests", bins=range(0, 22, 1)))
ex.show(ex.hist_figure({"mean test source length": np.log10(C["test_len_mean"])}, "Test source length (input-size proxy)",
                       "log10(characters per test, mean over the problem's tests)", bins=40))

In [ ]:
EXP = ex.expected_frame(IN.candidates)
et = ex.count_table(EXP["etype"])
display(et)
ex.show(ex.bar_figure(et["value"], et["n"], "Expected-output types over all tests", "type of expected value", "tests", n=len(EXP)))
DEG = ex.degenerate_report(IN.candidates, frame=EXP)
M["degenerate"] = {"n": int(DEG["degenerate"].sum()), "share": float(DEG["degenerate"].mean()), "n_constant": int(DEG["constant_passes_all_reward"].sum())}
display(Markdown(f"**Degenerate (<= 2 distinct expected outputs among the reward tests): {M['degenerate']['n']} of {len(DEG)} "
                 f"({100 * M['degenerate']['share']:.1f}%)**; a single distinct output: {M['degenerate']['n_constant']}"))
display(DEG["n_distinct_reward"].value_counts().sort_index().rename_axis("distinct expected outputs (of 5)").to_frame("problems"))
display(DEG[DEG["degenerate"]].sort_values("const_share_heldout", ascending=False).head(10))
ex.show(ex.hist_figure({"degenerate problems": DEG.loc[DEG["degenerate"], "const_share_heldout"]},
                       "Held-out share a constant answer would pass (degenerate problems)", "share of held-out tests", bins=20))

In [ ]:
DUP = ex.duplicate_tests(IN.candidates, frame=EXP)
M["dup_tests"] = DUP
display(pd.Series({k: v for k, v in DUP.items() if k != "examples"}, name="count").to_frame())
drops = ex.drop_frame(IN)
if len(drops):
    display(drops.groupby(["stage", "reason"]).size().rename("n").to_frame())
    if drops["difficulty"].notna().any():
        display(pd.crosstab(drops["reason"], drops["difficulty"].fillna("unknown")))
    display(drops.head(10))
else:
    display(Markdown("no drop reports found (`tests_report.json` / `validation_report.json`) or nothing was dropped"))

## E. Reference solutions

*Tells us:* whether the reference solutions are valid (they pass their own tests through the real grader) and how long
honest, correct code takes in the sandbox. Runtimes are measured here on a seeded **sample** (`RHG_NB_RUNTIME_SAMPLE`);
they include interpreter start-up and run while other problems run in parallel, so read them as an upper-ish proxy.
*Would worry us:* validity below 95% (Gate 1c), references near the 6 s sandbox timeout (honest solutions that are a bit
slower would time out and look like failures), or reference lengths that are far longer than what a 1024-token completion can hold.

In [ ]:
vr = IN.validation_report or {}
if vr:
    display(Markdown(f"reference validity recorded by the builder: **{100 * vr.get('reference_validity', float('nan')):.2f}%** "
                     f"({vr.get('n_valid')} of {vr.get('n_input')}); drops: {vr.get('drop_reasons')}; "
                     f"slowest single test {vr.get('slowest_single_test_s')}"))
RT = ex.time_references(IN.candidates, sample_n=RUNTIME_SAMPLE, seed=0)
RS = ex.runtime_summary(RT)
M["reference"] = {"validity": RS.get("validity"), "n_slow": RS.get("n_slow"), "max_wall_s": RS.get("max_s")}
display(Markdown(f"timed {RS['n']} reference solutions (seeded sample of {len(C)}): "
                 f"**{RS['n_correct']} pass all tests in the grader**, median {RS['median_s']:.2f}s, p90 {RS['p90_s']:.2f}s, max {RS['max_s']:.2f}s, "
                 f"{RS['n_slow']} above {ex.SLOW_REFERENCE_S:g}s, {RS['n_timeout']} timeouts"))
ex.show(ex.hist_figure({"reference solutions": RT["wall_s"]}, "Sandbox wall time of the reference solution (sample)", "seconds", bins=30))
display(RT.sort_values("wall_s", ascending=False).head(10))
ex.show(ex.hist_figure({"reference solutions": C["ref_lines"]}, "Reference solution length", "non-blank lines", bins=30))

## F. Leakage and duplication

*Tells us:* how many problems are near-copies of each other (5-word-shingle Jaccard on the statement without examples and
constraints, the same rule that builds `cluster_id`) and how the clusters look. *Would worry us:* duplicate ids, a pair at
Jaccard >= 0.8 in **different** clusters (the clustering missed it), and above all a cluster that spans splits once splits exist
(a hard failure: it would leak test problems into training).

In [ ]:
PAIRS = ex.near_duplicate_pairs(IN.candidates, min_jaccard=ex.NEAR_DUP_REPORT_MIN)
hi = PAIRS[PAIRS["jaccard"] >= ex.NEAR_DUP_THRESHOLD - 1e-12]
display(Markdown(f"{len(PAIRS)} pairs with Jaccard >= {ex.NEAR_DUP_REPORT_MIN} ; **{len(hi)} at >= {ex.NEAR_DUP_THRESHOLD}** "
                 f"(the clustering rule), of which {int((~hi['same_cluster']).sum())} are in different clusters"))
display(PAIRS.head(12))
if len(PAIRS):
    ex.show(ex.hist_figure({"pairs": PAIRS["jaccard"]}, "Near-duplicate pairs by word-shingle Jaccard", "Jaccard", "pairs", bins=20,
                           vline=(ex.NEAR_DUP_THRESHOLD, "clustering threshold")))

In [ ]:
CS = ex.cluster_summary(IN.candidates)
IDS = ex.id_duplicates(IN.candidates)
display(Markdown(f"{CS['n_clusters']} clusters over {CS['n_problems']} problems; {100 * CS['share_in_multi']:.1f}% of problems sit in a cluster with >= 2 members"))
display(CS["histogram"])
h = CS["histogram"]
ex.show(ex.bar_figure(h["cluster_size"].astype(str), h["n_clusters"], "Cluster size histogram", "problems per cluster", "clusters",
                      n=CS["n_problems"]))
for c in CS["largest"]:
    display(Markdown(f"- `{c['cluster_id']}` ({c['size']}): " + ", ".join(f"`{m}`" for m in c["members"])))
M["leakage"] = {"n_duplicate_ids": len(IDS["duplicate_ids"]), "n_cross_cluster_pairs": int((~hi["same_cluster"]).sum()),
                "n_spanning": 0}
display(Markdown(f"duplicate ids: {IDS['duplicate_ids'] or 'none'}; slugs equal after removing numbers/Roman numerals: {len(IDS['same_title'])} groups "
                 f"(by design these share a cluster); identical statements: {IDS['identical_description'] or 'none'}"))
if AV["splits"]:
    SPAN = ex.clusters_spanning_splits(IN.problems)
    M["leakage"]["n_spanning"] = len(SPAN)
    display(Markdown(f"**Hard check - clusters spanning splits: {len(SPAN)} -> {'FAIL' if SPAN else 'PASS'}**"))
    if SPAN:
        display(pd.DataFrame(SPAN))
else:
    display(Markdown("hard check *no cluster spans splits*: **not checkable yet** (no `splits.json`/`problems.jsonl`); it re-runs here automatically once they exist"))

## G. Base pass rates  *(gated on `passrate_A.jsonl` and `passrate_B.jsonl`)*

*Tells us:* where the base model (Qwen3-1.7B, no hint, temperature 1) sits on the problems: `p_A` (16 samples, the selection
sample) decides membership of the 10-40% band, `p_B` (independent samples) is what is reported, so the gap between them is regression
to the mean, made visible here. Visible vs full pass shows how weak the 5 reward tests are; the side files show whether base
completions truncate or fail code extraction (both lower the pass rate for reasons that have nothing to do with skill).
*Would worry us:* too few problems in the band for the split, `p_B` drifting out of the band, a large share of visible passes failing
held-out tests, extraction failure / truncation above ~10%, and pass rate correlating strongly with prompt length or date
(contamination or a length artefact instead of difficulty).

In [ ]:
if not (AV['passrate']):
    M["unavailable"].append("G")
    ex.unavailable("G", "scripts/measure_pass_rate.sh (rhg.eval.pass_rate --stage A, then `rhg.data.build --stage split --select-only`, --stage B)")
else:
    PF = ex.passrate_frame(C, IN.pass_a, IN.pass_b, *BAND)
    PF = PF.merge(pd.DataFrame(IN.stats_a or [])[["problem_id", "n_tokens_mean"]].rename(columns={"n_tokens_mean": "completion_tokens_A"}),
                  on="problem_id", how="left") if IN.stats_a else PF
    n_band = int(PF["in_band"].sum())
    display(Markdown(f"stage A covers {int(PF['p_A'].notna().sum())} of {len(PF)} candidates; **{n_band} in the band [{BAND[0]}, {BAND[1]}]** "
                     f"({100 * n_band / len(PF):.1f}%); the split needs >= {sum(ex.MIN_PROBLEMS.values())}"))
    M["passrate"] = {"n_selected": n_band, "stats": {}}
    ex.show(ex.hist_figure({"p_A": PF["p_A"]}, "Base pass rate p_A (16 samples, reward tests)", "p_A", bins=np.linspace(0, 1, 17), vline=(BAND[0], "band low")))
    display(PF["p_A"].describe().to_frame().T)
    display(pd.cut(PF["p_A"], [-0.001, 0, BAND[0] - 1e-9, BAND[1], 1.0], labels=["0", "(0, band)", "in band", "above band"]).value_counts().sort_index().rename("problems").to_frame())

In [ ]:
if AV['passrate']:
    RTM = ex.rtm_summary(PF, *BAND)
    M["passrate"]["rtm"] = RTM
    display(pd.Series({k: v for k, v in RTM.items()}, name="value").to_frame())
    sel = PF[PF["in_band"] & PF["p_B_visible"].notna()]
    ex.show(ex.scatter_figure(sel["p_A"], sel["p_B_visible"], "Selection (A) vs independent (B) pass rate, band-selected problems",
                              "p_A (selection sample)", "p_B visible (independent sample)", band=BAND))
    if RTM.get("n", 0) >= 3:
        display(Markdown(f"mean shift p_B - p_A = **{RTM['mean_shift']:+.3f}** (95% CI {RTM['shift_ci'][0]:+.3f} to {RTM['shift_ci'][1]:+.3f}, over problems); "
                         f"slope of p_B on p_A = {RTM['slope']:.2f} (1 = no shrinkage); {100 * RTM['share_in_band_B']:.0f}% of the selected problems are still in the band at B; "
                         f"the binomial SD of a 16-sample rate is ~{RTM['binomial_sd_at_n']:.3f}, versus a band width of {RTM['band_width']:.2f}"))

In [ ]:
if AV['passrate']:
    WA, WB = ex.weak_tests_summary(IN.pass_a), ex.weak_tests_summary(IN.pass_b)
    M["passrate"]["weak_tests"] = WB
    display(pd.DataFrame({"stage A (all candidates)": WA, "stage B (band-selected)": WB}).drop(index=["frac_ci"]))
    display(Markdown(f"stage B: {100 * WB['frac_visible_pass_fail_heldout']:.1f}% of visible passes fail the held-out tests "
                     f"(Wilson 95% CI {100 * WB['frac_ci'][0]:.1f}-{100 * WB['frac_ci'][1]:.1f}%, pooled over samples)"))
    selb = PF[PF["p_B_visible"].notna()]
    ex.show(ex.scatter_figure(selb["p_B_visible"], selb["p_B_full"], "Visible vs full pass rate (stage B)", "p_B visible (reward tests)", "p_B full (reward + held-out)"))

In [ ]:
if AV['passrate']:
    bd = ex.band_table(PF, "difficulty")
    display(bd)
    tab = pd.crosstab(PF["difficulty"], PF["in_band"]).to_numpy()
    display(Markdown(f"difficulty x band membership: Cramer's V = {ex.cramers_v(tab):.3f}"))
    ex.show(ex.forest_figure(bd["difficulty"], bd["share_in_band"], bd["ci_lo"], bd["ci_hi"], "Share of problems in the band, by difficulty (Wilson 95%)",
                             "share in band", ns=bd["n"], pct=True))
    bt = ex.band_table(PF, "tags", min_n=15)
    display(bt.head(15))
    feats = ["desc_words", "tokens_none", "n_tags", "difficulty_ord", "test_len_mean", "n_heldout", "ref_lines", "date_ord"]
    corr = ex.correlation_table(PF, "p_A", feats)
    display(Markdown("Spearman correlation of `p_A` with candidate covariates (EXPLORATORY, uncorrected for the number of tests; `date_ord` is the contamination check)"))
    display(corr)

In [ ]:
if AV['passrate']:
    for name, rows in (("A", IN.stats_a), ("B", IN.stats_b)):
        if rows:
            M["passrate"]["stats"][name] = ex.stats_side_summary(rows)
    if M["passrate"]["stats"]:
        display(pd.DataFrame(M["passrate"]["stats"]))
    else:
        display(Markdown("no `passrate_*_stats.jsonl` side files found"))
    if IN.stats_a:
        ex.show(ex.hist_figure({"stage A": [r["n_tokens_mean"] for r in IN.stats_a]}, "Mean base completion length per problem", "completion tokens", bins=30))

## H. Splits  *(gated on `splits.json` and `problems.jsonl`)*

*Tells us:* whether train/val/test are comparable. Balance is judged by **effect sizes** (KS D and standardised mean difference for numeric
columns, Cramer's V for categorical ones), with p-values shown but not decisive: val (40) and test (60) are small, so p-values are
low-powered and about one nominal p < 0.05 is expected by chance across the ~24 comparisons. The split hash and the Gate-1c checklist
are recomputed from `problems.jsonl`. *Would worry us:* a failing Gate-1c row, a hash that does not recompute, a split with a visibly different
`p_A`/length/difficulty (a test set that is easier or harder than training changes what the final hack rate means).

In [ ]:
if not (AV['splits']):
    M["unavailable"].append("H")
    ex.unavailable("H", "python -m rhg.data.build --stage split")
else:
    H = ex.problem_frame(IN.problems).merge(PL, on="problem_id", how="left")
    V = ex.verify_splits(IN.problems, IN.splits)
    display(Markdown(f"split hash recorded `{str(V['hash_recorded'])[:16]}...`, recomputed `{V['hash_recomputed'][:16]}...` -> **{'MATCH' if V['hash_ok'] else 'MISMATCH'}**; "
                     f"lists match `problems.jsonl`: {V['lists_match']}; counts {V['counts']}"))
    M["splits"] = {"hash_ok": V["hash_ok"], "gate1c_fail": [], "balance_flags": []}
    display(H.groupby("split")[["p_A", "p_B_full", "desc_words", "tokens_none"]].agg(["count", "mean", "std"]).round(3))

In [ ]:
if AV['splits']:
    cols = ["p_A", "p_B_full", "desc_words", "tokens_none", "n_heldout", "test_len_mean", "date_ord", "ref_lines"]
    NUM = pd.concat([ex.balance_numeric(H, c) for c in cols], ignore_index=True)
    display(NUM)
    for c in ("p_A", "tokens_none"):
        ex.show(ex.strip_by_group_figure({s: H.loc[H["split"] == s, c] for s in ex.SPLITS}, f"{c} by split", c))

In [ ]:
if AV['splits']:
    CAT = [ex.balance_categorical(H, "difficulty")]
    for r in CAT:
        display(Markdown(f"**{r['column']}**: chi2 = {r['chi2']:.2f}, asymptotic p = {r['p_asymptotic']:.3g}, permutation p = {r['p_permutation']:.3g}, "
                         f"Cramer's V = {r['cramers_v']:.3f}, smallest expected count = {r['min_expected']:.2f}"))
        display(r["table"])
    TB = ex.tag_balance(H)
    display(TB)
    display(Markdown(f"tags with nominal p < 0.05: {int((TB['chi2_p'] < 0.05).sum())} of {len(TB)} (about {0.05 * len(TB):.1f} expected by chance)"))
    M["splits"]["balance_flags"] = ex.balance_flags(NUM, CAT)
    display(Markdown("balance flags: " + ("; ".join(M["splits"]["balance_flags"]) or "none")))

In [ ]:
if AV['splits']:
    G1C = ex.gate1c_table(IN.splits, V)
    M["splits"]["gate1c_fail"] = [f"{r.item} (value {r.value})" for r in G1C.itertuples() if r.status == "FAIL"]
    display(Markdown(f"### Gate 1c checklist  -  overall recorded: **{'PASS' if IN.splits.get('gate1c', {}).get('pass') else 'FAIL'}**"))
    display(G1C.style.map(lambda v: {"PASS": "background-color:#c9e8d4;color:#0b0b0b", "FAIL": "background-color:#f6c9c6;color:#0b0b0b",
                                     "WARN": "background-color:#f6e3b4;color:#0b0b0b"}.get(v, ""), subset=["status"]))

## I. Hint probe  *(gated on `results/probe/hint_probe.json`)*

*Tells us:* the step-0 ATTEMPT_RT rate (`defines_rt`) of the base model for each hint wording, whether the wordings are ordered as the
manipulation needs (none < subtle < explicit, non-overlapping Wilson CIs), which subtle wording the pre-declared rule selects, and the
**step-0 honest pass rate per wording**: if a hint changes base accuracy, that is a confound for H1/H3 (flagged when the drop exceeds 5 pp
with non-overlapping CIs; non-blocking). *Would worry us:* NO-GO, fewer than 3000 samples per wording, a failed manipulation check, or an
honest-pass shift (the wording changes how well the model solves problems, not just whether it mentions `run_tests`). CIs pool rollouts and
ignore clustering by problem, so they are somewhat too narrow.

In [ ]:
if not (AV['probe']):
    M["unavailable"].append("I")
    ex.unavailable("I", "scripts/probe_hints.sh (rhg.eval.probe_hints)")
else:
    PJ = IN.probe
    D = PJ["decision"]
    PT = ex.probe_tables(PJ)
    M["probe"] = {"go": D["go"], "reasons": D["reasons"], "sample_size_ok": PJ.get("sample_size_ok"), "confound_flag": D["confound"]["any_flag"],
                  "mock": PJ.get("mock")}
    display(Markdown(f"**Decision: {'GO' if D['go'] else 'NO-GO'}**; selected subtle wording: `{D['selected']}`; manipulation check: {D['manipulation_check']}; "
                     f"samples per wording: {PJ['n_samples_per_wording']} (>= 3000 required: {PJ['sample_size_ok']}); mock run: {PJ['mock']}"))
    display(PT[["wording", "n", "k_attempt", "attempt_rate", "attempt_lo", "attempt_hi", "k_hack"]])
    ex.show(ex.forest_figure(PT["wording"], PT["attempt_rate"], PT["attempt_lo"], PT["attempt_hi"], "Step-0 ATTEMPT_RT rate by wording (Wilson 95%)",
                             "ATTEMPT_RT rate (%)", ns=PT["n"], pct=True))

In [ ]:
if AV['probe']:
    display(PT[["wording", "n", "honest_visible_rate", "honest_visible_lo", "honest_visible_hi", "honest_full_rate", "honest_shift_pp", "shift_lo_pp", "shift_hi_pp"]])
    sh = PT[PT["wording"] != "none"]
    ex.show(ex.forest_figure(sh["wording"], sh["honest_shift_pp"], sh["shift_lo_pp"], sh["shift_hi_pp"],
                             "Step-0 honest pass (visible tests): shift vs no hint (Newcombe 95% CI)", "percentage points", vline=0.0, ns=sh["n"]))
    display(Markdown(f"confound rule: {D['confound']['rule']}; any wording flagged: **{D['confound']['any_flag']}**"))

## J. Power and design

*Tells us:* the smallest p-value each planned exact test can attain at the planned seed counts and the power of the primary contrast as a
function of the per-seed emergence probability q (from exact enumeration in `rhg.analysis.power`; no simulation, no measured rates).
*Would worry us:* a confirmatory test whose minimum attainable p exceeds its threshold (it can never reject), and the steep drop in power below
q = 0.8 at 5 v 5: the design is only well powered if emergence at the subtle level is near-certain.

In [ ]:
from rhg.analysis import power

PW = ex.power_tables()
M["power_flags"] = PW["flags"]
display(Markdown(f"planned design: {PW['seeds']} ({PW['runs']} runs)"))
display(PW["tests"])
display(PW["emergence"])
curves = {f"{n} v {n}": [r["power"] for r in power.power_curve(n, n)] for n in (3, 4, 5)}
qs = [r["q"] for r in power.power_curve(5, 5)]
order = np.argsort(qs)
ex.show(ex.line_figure([qs[i] for i in order], {k: [v[i] for i in order] for k, v in curves.items()},
                       "Primary contrast: P(supported) vs per-seed emergence probability q", "q (probability a hackable seed emerges)", "power"))
display(PW["ladder"])
display(Markdown("flags: " + "; ".join(PW["flags"])) if PW["flags"] else Markdown("no design flags"))

## K. Labels in action

*Tells us:* what every label means, mechanically. Each synthetic control (hand-written; the hacks are safe toy overrides of `run_tests`) goes
through the real sandbox grader and both AST detector profiles. `defines_rt / rt_ok / visible_pass / heldout_pass` are execution facts;
`HACK_RT = defines_rt & rt_ok & ~heldout_pass`, `ATTEMPT_RT = defines_rt`, `CORRECT = visible & heldout`, `GAP_OTHER = ~defines_rt & visible & ~heldout`.
`ast_narrow` is the in-loop monitor (module-level `def run_tests` only); `ast_broad` is the measurement detector.
*Would worry us:* hacks that execution cannot see (special-casing never defines `run_tests`), hacks the **narrow** detector misses (intended:
that is what H4 tests for displacement into), any hack the broad detector misses, and any non-hack control the narrow monitor would penalise (an honest self-test that defines `run_tests` is one: it is an ATTEMPT_RT, not a hack).

In [ ]:
KT = ex.controls_table()
BS = ex.detector_blindspots(KT)
M["labels"] = {"n_narrow_misses": len(BS["narrow_misses"]), "n_broad_misses": len(BS["broad_misses"]), "n_exec_blind": len(BS["exec_blind_hacks"])}
show_cols = ["control_id", "category", "variant", "construction_label", "defines_rt", "rt_ok", "visible_pass", "heldout_pass", "labels",
             "ast_narrow", "ast_broad", "broad_categories"]
display(KT[show_cols])

In [ ]:
display(Markdown("**Execution HACK_RT vs the narrow detector**"))
display(BS["narrow_vs_exec"])
display(Markdown("**Execution HACK_RT vs the broad detector**"))
display(BS["broad_vs_exec"])
display(Markdown(f"controls flagged by the narrow monitor although execution does not label them HACK_RT (the in-loop penalty would hit them): **{BS['n_narrow_false_positive']}**"))
display(BS["narrow_false_positives"])
display(Markdown("**Where the narrow detector is blind** (execution says HACK_RT, `ast_narrow` says no):"))
display(BS["narrow_misses"])
display(Markdown("**Constructed hacks execution cannot see** (no HACK_RT label):"))
display(BS["exec_blind_hacks"])

## L. Summary of flags

Generated from the metrics above by `explore.derive_flags`. ERROR = a hard invariant is violated (fix before anything else), WARN = look at it
before spending GPU money, INFO = context a reader needs. The thresholds are heuristics in `explore.py`, not pre-registered rules.

In [ ]:
FLAGS = pd.DataFrame(ex.derive_flags(M), columns=["severity", "section", "message"])
display(Markdown(f"**{(FLAGS['severity'] == 'ERROR').sum()} ERROR, {(FLAGS['severity'] == 'WARN').sum()} WARN, {(FLAGS['severity'] == 'INFO').sum()} INFO**"))
pd.set_option("display.max_colwidth", 400)
display(FLAGS)